# Agile Autonomy — Model Training on Google Colab

This notebook trains the **PlaNet** trajectory-prediction network from the
[Learning High-Speed Flight in the Wild](http://rpg.ifi.uzh.ch/AgileAutonomy.html) paper.

**Before you start**:
1. Go to **Runtime → Change runtime type** and select **GPU** (T4 is free; V100/A100 are faster on Colab Pro).
2. Run the cells top-to-bottom in order.
3. *(Optional)* Mount Google Drive (Step 3) so checkpoints survive a runtime reset.

**What you need**:
- A Zenodo account is **not** required — the dataset is publicly available.
- The dataset download is ~6 GB; allow 10–20 minutes depending on your connection.
- Full training (100 epochs, default config) takes ~3–6 hours on a T4 GPU.

## Step 1 — Verify GPU

In [ ]:
import subprocess, sys

# Show GPU info
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

# Show CUDA version (determines which TensorFlow to install)
cuda_out = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
print(cuda_out)

import re
match = re.search(r'release (\d+)\.', cuda_out)
cuda_major = int(match.group(1)) if match else 0
print(f'Detected CUDA major version: {cuda_major}')

## Step 2 — Install Python Dependencies

No ROS installation is required for offline supervised training.

In [ ]:
import subprocess, sys

# Detect CUDA version and pick a compatible TensorFlow
cuda_out = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
import re
match = re.search(r'release (\d+)\.', cuda_out)
cuda_major = int(match.group(1)) if match else 12

if cuda_major >= 12:
    # Colab with CUDA 12.x — TF 2.15+ required
    TF_PACKAGE = 'tensorflow[and-cuda]==2.15.0'
elif cuda_major == 11:
    # Colab with CUDA 11.x — TF 2.12 is the last to bundle CUDA 11 wheels
    TF_PACKAGE = 'tensorflow-gpu==2.12.0'
else:
    TF_PACKAGE = 'tensorflow-gpu==2.4.0'

print(f'Installing {TF_PACKAGE} ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', TF_PACKAGE], check=True)

# Core training dependencies (no ROS)
DEPS = [
    'open3d>=0.13',       # pointcloud KDTree for collision-aware loss
    'pyquaternion',       # quaternion → rotation matrix
    'opencv-python-headless',
    'scipy',
    'pandas',
    'tqdm',
    'pyyaml',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS, check=True)
print('All dependencies installed.')

## Step 3 — (Optional) Mount Google Drive

Skip this cell if you do not want persistent storage.
If you mount Drive, the dataset and checkpoints are cached across sessions.

In [ ]:
USE_DRIVE = False   # <-- set to True to enable Drive

DRIVE_ROOT = '/content/drive/MyDrive/agile_autonomy'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f'Drive mounted. Working directory: {DRIVE_ROOT}')
else:
    DRIVE_ROOT = '/content'
    print('Drive not mounted. Files will be lost on runtime reset.')

## Step 4 — Clone the Repository

In [ ]:
import os

REPO_DIR  = '/content/agile_autonomy'
TRAIN_DIR = os.path.join(REPO_DIR, 'planner_learning')

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/rayeed221/agile_autonomy.git {REPO_DIR}
else:
    print('Repo already cloned, pulling latest changes ...')
    !git -C {REPO_DIR} pull

print('Repository ready at:', REPO_DIR)

## Step 5 — Download the Agile Autonomy Dataset

The dataset is hosted on Zenodo (record 5517791) and was collected at ~7 m/s.
It contains depth images, odometry CSVs, trajectory labels, and 3-D point clouds.

> **Size**: ~6 GB compressed. Extraction yields ~20 GB on disk.  
> If you have Google Drive mounted, the download is skipped on repeated runs.

In [ ]:
import os

DATASET_URL  = 'https://zenodo.org/record/5517791/files/agile_autonomy_dataset.tar.xz?download=1'
DATASET_ARCHIVE = os.path.join(DRIVE_ROOT, 'agile_autonomy_dataset.tar.xz')
DATASET_DIR     = os.path.join(DRIVE_ROOT, 'dataset')

DATA_TRAIN = os.path.join(DATASET_DIR, 'train')
DATA_VAL   = os.path.join(DATASET_DIR, 'val')

if not os.path.isdir(DATA_TRAIN):
    if not os.path.isfile(DATASET_ARCHIVE):
        print('Downloading dataset (~6 GB) ...')
        !wget -q --show-progress -O "{DATASET_ARCHIVE}" "{DATASET_URL}"
    else:
        print('Archive already on disk, skipping download.')

    print('Extracting archive (this may take a few minutes) ...')
    os.makedirs(DATASET_DIR, exist_ok=True)
    !tar -xf "{DATASET_ARCHIVE}" -C "{DATASET_DIR}" --strip-components=1
    print('Extraction complete.')
else:
    print('Dataset already extracted at:', DATASET_DIR)

# Verify expected structure
for split in ['train', 'val']:
    split_dir = os.path.join(DATASET_DIR, split)
    rollouts  = [d for d in os.listdir(split_dir) if d.startswith('rollout')] if os.path.isdir(split_dir) else []
    print(f'  {split}: {len(rollouts)} rollout(s) found')

## Step 6 — Write Cloud Training Config

A copy of `train_settings.yaml` is written with absolute paths so training
works from any working directory.

In [ ]:
import os, yaml

LOG_DIR        = os.path.join(DRIVE_ROOT, 'checkpoints')
CONFIG_SRC     = os.path.join(TRAIN_DIR, 'config', 'train_settings.yaml')
CONFIG_CLOUD   = os.path.join(TRAIN_DIR, 'config', 'train_settings_colab.yaml')

os.makedirs(LOG_DIR, exist_ok=True)

with open(CONFIG_SRC) as f:
    cfg = yaml.safe_load(f)

# Override paths for Colab
cfg['log_dir']           = LOG_DIR
cfg['train']['train_dir'] = DATA_TRAIN
cfg['train']['val_dir']   = DATA_VAL

# Recommended Colab tweaks
cfg['train']['batch_size']          = 16   # T4 has 16 GB — increase if you have a bigger GPU
cfg['train']['max_training_epochs'] = 100
cfg['train']['save_every_n_epochs'] = 5
cfg['train']['summary_freq']        = 200

with open(CONFIG_CLOUD, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config written to:', CONFIG_CLOUD)
print(yaml.dump(cfg, default_flow_style=False))

## Step 7 — Run Training

`train.py` must be launched from the `planner_learning/` directory because it
uses relative imports and copies `nets.py` into the log dir for reproducibility.

In [ ]:
import os, subprocess, sys

TRAIN_DIR   = '/content/agile_autonomy/planner_learning'
CONFIG_FILE = os.path.join(TRAIN_DIR, 'config', 'train_settings_colab.yaml')

os.chdir(TRAIN_DIR)
print('Working directory:', os.getcwd())

cmd = [sys.executable, 'train.py', f'--settings_file={CONFIG_FILE}']
print('Running:', ' '.join(cmd))

# Stream output live
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode == 0:
    print('\nTraining finished successfully.')
else:
    print(f'\nTraining exited with code {process.returncode}.')

## Step 8 — Monitor Training with TensorBoard

Run this cell at any time (including while training is running in another cell
via `%%bash` or a background thread) to inspect loss curves.

In [ ]:
import os
LOG_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')   # same as Step 6

%load_ext tensorboard
%tensorboard --logdir "{LOG_DIR}"

## Step 9 — Resume Training from a Checkpoint

Edit and run this cell to continue a previous run.

In [ ]:
import os, yaml

TRAIN_DIR    = '/content/agile_autonomy/planner_learning'
CONFIG_CLOUD = os.path.join(TRAIN_DIR, 'config', 'train_settings_colab.yaml')

# List available checkpoints
LOG_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
for root, dirs, files in os.walk(LOG_DIR):
    for f in files:
        if f.endswith('.index'):
            ckpt_path = os.path.join(root, f.replace('.index', ''))
            print(ckpt_path)

# Set the checkpoint you want to resume from (copy one of the paths above)
RESUME_CKPT = ''   # e.g. '/content/drive/MyDrive/agile_autonomy/checkpoints/20240101-120000/train/ckpt-10'

if RESUME_CKPT:
    with open(CONFIG_CLOUD) as f:
        cfg = yaml.safe_load(f)
    cfg['checkpoint']['resume_training'] = True
    cfg['checkpoint']['resume_file']     = RESUME_CKPT
    with open(CONFIG_CLOUD, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print('Config updated to resume from:', RESUME_CKPT)
else:
    print('No checkpoint set — will train from scratch.')

## Step 10 — Export Trained Model

Converts the best checkpoint to a TensorFlow SavedModel (and optionally TFLite)
for deployment.

In [ ]:
import os, glob, subprocess, sys

TRAIN_DIR   = '/content/agile_autonomy/planner_learning'
EXPORT_DIR  = os.path.join(DRIVE_ROOT, 'exported_model')
os.makedirs(EXPORT_DIR, exist_ok=True)

# Find the latest checkpoint directory
LOG_DIR  = os.path.join(DRIVE_ROOT, 'checkpoints')
run_dirs = sorted(glob.glob(os.path.join(LOG_DIR, '*', 'train')))
if not run_dirs:
    print('No checkpoint directories found. Run Step 7 first.')
else:
    LATEST_CKPT_DIR = run_dirs[-1]
    # Read the latest checkpoint name
    ckpt_file = os.path.join(LATEST_CKPT_DIR, 'checkpoint')
    with open(ckpt_file) as f:
        first_line = f.readline().strip()
    ckpt_name = first_line.split('"')[1]
    CKPT_PATH = os.path.join(LATEST_CKPT_DIR, ckpt_name)
    print('Exporting from checkpoint:', CKPT_PATH)

    os.chdir(TRAIN_DIR)

    # Locate the settings file used for this run
    SETTINGS_IN_CKPT = glob.glob(os.path.join(run_dirs[-1], '..', '*.yaml'))
    SETTINGS_FILE    = SETTINGS_IN_CKPT[0] if SETTINGS_IN_CKPT else os.path.join(TRAIN_DIR, 'config', 'train_settings_colab.yaml')

    cmd = [
        sys.executable, 'export_model.py',
        f'--settings_file={SETTINGS_FILE}',
        f'--checkpoint={CKPT_PATH}',
        f'--out_folder={EXPORT_DIR}',
    ]
    subprocess.run(cmd, check=True)
    print('Model exported to:', EXPORT_DIR)